In [81]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import re
import pandas as pd
import numpy as np
import requests
import math
from collections import defaultdict, Counter

<h4>Necessary imports: regular expressions,pandas, the math library requests, the defaultdict data structure, Counter and necesseties from sklearn

In [82]:
WIKI_API = "https://en.wikipedia.org/w/api.php"

DOC_TITLES = [
    "Baseball",
    "Babe Ruth",
    "Willie Mays",
    "Ted Williams",
    "Hank Aaron",
    "Jackie Robinson",
    "Sandy Koufax",
    "Basketball",
    "Michael Jordan",
    "LeBron James",
    "Omri Caspi",
    "Magic Johnson",
    "Gal Mekel" 
    "Swimming",
    "Michael Phelps",
    "Johnny Weissmuller",
    "Mark Spitz",
    "Katie Ledecky",
    "Ian Thorpe",
    "Dawn Fraser"
]
HEADERS = {
    "User-Agent": "Python/requests"
}

<h4>Constants defenition - Wikipedia's API, page titles and request necessaties</h4>

In [83]:
def wiki(title):
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": False,
        "titles": title,
        "format": "json",
        "redirects": 0,
        "formatversion": 2,
        "exintro" : 1
    }
    response = requests.get(WIKI_API, params=params, headers=HEADERS)
    pages = response.json().get("query", {}).get("pages", [])
    if not pages:
        return ""
    return pages[0].get("extract", "")

<h4>A function to handle and execute Wikipedia's API calls</h4>

In [84]:
def freq(titles):
    documents = {}
    countings = {}
    rows = []
    vocabulary = set()
    for title in titles:
        text = wiki(title) # fetch page
        tokens =[t for t in (re.split(r"\W+", text.lower())) if t.isalpha()] # tokenization
        count = Counter(tokens) # count token in page
        countings[title] = count #keep number of tokens for page
        vocabulary.update(count.keys())
    vocabulary = sorted(vocabulary)
    for title in titles:
        row = [countings[title].get(term,0) for term in vocabulary] # build rows for dataframe
        rows.append(row)
    df = pd.DataFrame(rows, index=titles, columns=vocabulary)
    return df

In [85]:
def getmetrics(df, k1=1.6, b=0.75):
    N = df.shape[0]
    doc_lens = df.sum(axis=1).astype(float)                # get length per doc
    avgdl = doc_lens.mean()
    df_term = (df > 0).sum(axis=0).to_dict() # calculate document frequency for each term 
    idf = {term: math.log((N) / (df_term[term])) for term in df.columns} 
    return {"df": df_term, "idf": idf, "doc_lens": doc_lens, "avgdl": avgdl, "k1": k1, "b": b, "dataframe": df}

In [86]:
def bm25dataframe(metrics, query_terms):
    df, idf, k1, b, avgdl, doc_lens = metrics["dataframe"], metrics["idf"], metrics["k1"], metrics["b"], metrics["avgdl"], metrics["doc_lens"]
    denomenator_base = k1 * (1 - b + b * (doc_lens / avgdl))
    contribution = pd.DataFrame(0.0, index=df.index, columns=query_terms)
    for term in query_terms:
        if term in df.columns:
            tf = df[term].astype(float)
            denomenator = tf + denomenator_base
            contribution[term] = idf.get(term, 0.0) * (tf * (k1 + 1) / denomenator)
        else:
            contribution[term] = 0.0
    return contribution


<h4>All the above functions were fetched from the previous assignment (2) and adapted as needed to comly with the requirements of this assignment</h4>

In [93]:
freqdf = freq(DOC_TITLES)
df = bm25dataframe(getmetrics(freqdf),freqdf.columns)

<h4>sklearn's Kmeans only takes Euclidean distance as the Distance metric. Since we were specifically asked to consider Cosine Similarity as the distance metric, we'll explore the interesting link between Euclidean Distance and Cosine Similarity, that will allow us to comply with the requirements of the assignment, and utilize sklearn's Kmeans: </h4>

<strong>Sklearn's Kmeans minimizes the Euclidean Distance ${\| u-v \|^2}$ when ${u}$ and ${v}$ are two feature vecotors.<br>
w.l.o.g in our case (${u}$ and ${v}$ are feature vectors), assuming that ${u}$ and ${v}$ are column vectors:<br>
${\| u-v \|^2=(u-v)^T(u-v)}$ because of the squared Euclidean norm identity. <br>
if we expand the right side we get: <br>
${(u-v)^T(u-v) = u^Tu-2u^Tv+v^Tv}$.<br>
if the vectors ${u}$, ${v}$ were normalized, i.e ${\| u \|=\| v \|=1}$, combined with the identity ${x^Tx = \| x \|^2}$ for column vectors, the below equation would hold:<br>
${u^Tu-2u^Tv+v^Tv = 2-2u^Tv}$.<br>
Finally, using the base identity for unit vectors ${u^Tv = \cos\angle(u,v)}$, we get overall:<br>
${\| u-v \|^2= 2-2\cos\angle(u,v)}$.<br>
Thus, minimizing the squared Euclidean Distance between unit vectors equals to maximizing the cosine similarity. <br>
Conclusion: we need to normalize our dataset using L2 in order to use Sklearn's Kmeans and meet the demand of using cosine similarity as the distance metric. </strong>

In [88]:
df_mormalized = normalize(df, norm='l2', axis=1)

In [89]:
kmeans = KMeans(n_clusters=3, random_state=0, n_init=10)
kmeans.fit(df_mormalized)
labels = kmeans.labels_
centroids = kmeans.cluster_centers_
centroids_normilized = normalize(centroids, norm='l2', axis=1) #since we've normalized the dataset to comply with cosine similarity, we need to normalize the centroids themselves to in order to present the correct significant features

C:\Users\USER\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [94]:
results = []
for cluster in range(3):
    indicies = np.argsort(-centroids_normilized[cluster][:10])
    topterms = [freqdf.columns[i] for i in indicies]
    centroids_components = centroids_normilized[cluster][indicies]
    size = int((labels==cluster).sum())
    results.append({'cluster' : cluster,
                'top_terms' : topterms,
                'centroid_components' : centroids_components,
                'size' : size})
rows = []
for result in results:
    row = {'cluster' : result['cluster'], 'size' : result['size']}
    for i, (term, component) in enumerate(zip(result['top_terms'], result['centroid_components']), start=1):
        row[f'term_{i}'] = term
        row[f'centroid component_{i}'] = float(component)
    rows.append(row)
df_out = pd.DataFrame(rows).sort_values("cluster").reset_index(drop=True)

In [95]:
df_out

,cluster,size,term_1,centroid component_1,term_2,centroid component_2,term_3,centroid component_3,term_4,centroid component_4,...,term_6,centroid component_6,term_7,centroid component_7,term_8,centroid component_8,term_9,centroid component_9,term_10,centroid component_10
0,0,7,aau,0.056919,abilities,0.030067,accumulate,0.030067,a,0.016755,...,about,0.000000,abruptly,0.000000,accolades,0.000000e+00,achieve,0.000000e+00,aaron,-3.613822e-18
1,1,2,a,0.009792,aaron,0.000000,aau,0.000000,abilities,0.000000,...,about,0.000000,abruptly,0.000000,accolades,0.000000e+00,accumulate,0.000000e+00,achieve,0.000000e+00
2,2,10,aaron,0.067502,ability,0.036953,about,0.025616,abruptly,0.021257,...,achieve,0.021257,a,0.017049,abilities,2.376158e-18,accumulate,2.376158e-18,aau,0.000000e+00


<h4>the top 10 terms and corresponding (normalized) centroid components are presented in the dataframe above - as requested. Judging by the size of the clusters and the initial distribution of documents and themes, BM25 proves to be an ineffective metric with KMeans when the distance metric is cosine similarity in terms of correctly differentiating and clustering documents with same general theme</h4>